# Adding ClinVar benign variants as additional negatives

Goal: expand the negative class (previously only 250 OncoKB negatives, 35 in complete-case).

Label rule:
- Positive (1): OncoKB Oncogenic or Likely Oncogenic
- Negative (0): OncoKB Likely Neutral OR ClinVar Benign / Likely benign

Restricted to missense variants, since the tools only score missense.
Note: VEST4, REVEL, MutPred, VARITY were trained on ClinVar, so their scores are inflated against ClinVar benigns.

## 1. Load the OncoKB-annotated data

Loads the exonic variant file (tool scores plus OncoKB oncogenicity) and keeps only rows OncoKB successfully annotated. Defines the 9 tool-score columns used throughout.

In [11]:
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_csv("exonic_toolscores_oncokb_apicall.txt", sep="\t", low_memory=False)
df = df[df["ANNOTATED"] == True]

tool_cols = ["VEST4_score","REVEL_score","MutPred_score","PrimateAI_score",
             "VARITY_R_score","VARITY_ER_score","ESM1b_score","EVE_score","AlphaMissense_score"]
print("rows:", len(df))

rows: 169360


## 2. Load ClinVar classifications

Loads the ClinVar file checks the label distribution. Confirms usable benign labels are present: 764 Benign, 1,183 Likely_benign, and 684 Benign/Likely_benign (about 2,600 clean benign in total). Uncertain and Conflicting entries are not usable and are excluded later.

In [12]:
clinvar = pd.read_csv("hersh_exome_biallelic_annovar_annotated.hg38_multianno.missense_ref_alt_chrom_position_gene_clinvar.tsv", sep="\t", low_memory=False)
print(clinvar.columns.tolist())
print(clinvar["ClinVar_CLNSIG"].value_counts(dropna=False))

['Ref', 'Alt', 'Chrom', 'Position', 'Gene', 'ClinVar_CLNSIG']
ClinVar_CLNSIG
.                                                                       47210
Uncertain_significance                                                  41003
Conflicting_classifications_of_pathogenicity                            10983
Likely_benign                                                            1183
Benign                                                                    764
Benign/Likely_benign                                                      684
Pathogenic/Likely_pathogenic                                              265
Pathogenic                                                                239
Likely_pathogenic                                                         176
not_provided                                                               70
Uncertain_significance/Uncertain_risk_allele                               20
Likely_risk_allele                                               

## 3. Merge ClinVar onto the variants

Joins ClinVar significance onto each variant by chromosome, position, ref, and alt. All 102,603 missense variants matched a ClinVar entry, confirming the merge keys align (no chr-format or coordinate mismatch).

In [13]:
clv = clinvar[["Chrom","Position","Ref","Alt","ClinVar_CLNSIG"]].rename(
        columns={"Chrom":"Chr","Position":"Start"})

clv = clinvar[["Chrom","Position","Ref","Alt","ClinVar_CLNSIG"]].rename(
        columns={"Chrom":"Chr","Position":"Start"})
df = df.merge(clv, on=["Chr","Start","Ref","Alt"], how="left")
print("variants with a ClinVar call:", df["ClinVar_CLNSIG"].notna().sum())

variants with a ClinVar call: 102603


## 4. Build the hybrid label

Positive = OncoKB Oncogenic or Likely Oncogenic. Negative = OncoKB Likely Neutral or ClinVar Benign / Likely benign. Restricted to missense variants, since the tools only score missense.

Significance: Negatives expand from 219 (OncoKB only) to 2,751 (219 OncoKB + 2,585 ClinVar), against 824 positives. The earlier scarce-negative problem is resolved, and the class balance flips from positive-heavy to roughly 1 positive to 3 negatives.

In [14]:
benign = ["Benign","Likely_benign","Benign/Likely_benign"]

pos = df["ONCOGENIC"].isin(["Oncogenic","Likely Oncogenic"])
neg = (df["ONCOGENIC"] == "Likely Neutral") | (df["ClinVar_CLNSIG"].isin(benign))
missense = df["ExonicFunc.refGeneWithVer"] == "nonsynonymous SNV"

lab = df[missense & (pos | neg)].copy()
lab["label"] = pos[lab.index].astype(int)

for c in tool_cols:
    lab[c] = pd.to_numeric(lab[c], errors="coerce")

print(lab["label"].value_counts())
print("neg from OncoKB :", ((lab.label==0) & (lab["ONCOGENIC"]=="Likely Neutral")).sum())
print("neg from ClinVar:", ((lab.label==0) & (lab["ClinVar_CLNSIG"].isin(benign))).sum())

label
0    2751
1     824
Name: count, dtype: int64
neg from OncoKB : 219
neg from ClinVar: 2585


## 5. Per-tool performance (AUROC / AUPRC)

Each tool is scored on the labeled missense variants it covers. AUROC is the probability the tool ranks a random oncogenic variant above a random benign one.

Significance: with a real negative class, all AUROCs rise (0.76 to 0.87). The tools trained on ClinVar (VARITY, REVEL, VEST4, MutPred) rise the most, which reflects label leakage, since they are partly being tested on data they trained on. The fairest read is among tools not trained on ClinVar, where AlphaMissense leads.

In [15]:
results = []
for c in tool_cols:
    s = lab[[c, "label"]].dropna()
    scores = s[c].values
    if c == "ESM1b_score":
        scores = -scores
    y = s["label"].values
    results.append({
        "tool": c, "n": len(s),
        "n_pos": int(y.sum()), "n_neg": int((y == 0).sum()),
        "AUROC": round(roc_auc_score(y, scores), 3),
        "AUPRC": round(average_precision_score(y, scores), 3),
    })
res = pd.DataFrame(results).sort_values("AUROC", ascending=False).reset_index(drop=True)
print(res)

                  tool     n  n_pos  n_neg  AUROC  AUPRC
0       VARITY_R_score  3000    714   2286  0.865  0.734
1      VARITY_ER_score  3000    714   2286  0.840  0.688
2          REVEL_score  3428    813   2615  0.826  0.653
3  AlphaMissense_score  3491    814   2677  0.823  0.696
4          VEST4_score  3493    814   2679  0.821  0.645
5          ESM1b_score  3101    736   2365  0.810  0.635
6        MutPred_score  2031    707   1324  0.797  0.714
7            EVE_score  2180    619   1561  0.779  0.643
8      PrimateAI_score  3338    791   2547  0.761  0.516


## 6. Fair comparison on identical variants

Restricts to variants scored by every tool (complete cases) so all tools are ranked on the same set, removing coverage differences as a confound.

Significance: the complete-case set is now 1,324 variants with 783 negatives, versus only 35 negatives before ClinVar, so this ranking is finally well-powered. VARITY_R leads (0.860), but among tools not trained on ClinVar, AlphaMissense (0.821) is the standout.

In [16]:
complete = lab.dropna(subset=tool_cols).copy()
print("complete-case rows:", len(complete))
print(complete["label"].value_counts())

y = complete["label"].values
rows = []
for c in tool_cols:
    scores = complete[c].values
    if c == "ESM1b_score":
        scores = -scores
    rows.append({"tool": c,
                 "AUROC": round(roc_auc_score(y, scores), 3),
                 "AUPRC": round(average_precision_score(y, scores), 3)})
res_cc = pd.DataFrame(rows).sort_values("AUROC", ascending=False).reset_index(drop=True)
print(res_cc)

complete-case rows: 1324
label
0    783
1    541
Name: count, dtype: int64
                  tool  AUROC  AUPRC
0       VARITY_R_score  0.860  0.825
1      VARITY_ER_score  0.831  0.790
2          REVEL_score  0.822  0.785
3  AlphaMissense_score  0.821  0.798
4          VEST4_score  0.819  0.780
5          ESM1b_score  0.816  0.749
6        MutPred_score  0.805  0.767
7            EVE_score  0.793  0.751
8      PrimateAI_score  0.751  0.676


## 7. Recall at each tool's operating point

Assigns each tool a Youden-optimal cutoff and reports recall (fraction of oncogenic variants caught), specificity, and precision.

Significance: no single tool catches more than about 74 percent of oncogenic variants even at its best-balanced threshold, so roughly a quarter are missed by any one tool. 

In [17]:
from sklearn.metrics import roc_curve

rows = []
for c in tool_cols:
    s = lab[[c, "label"]].dropna()
    scores = s[c].values.astype(float)
    if c == "ESM1b_score":
        scores = -scores                      # flip so higher = more oncogenic
    y = s["label"].values

    fpr, tpr, thr = roc_curve(y, scores)
    k = (tpr - fpr).argmax()                  # Youden's J: maximize TPR - FPR
    t = thr[k]

    pred = (scores >= t).astype(int)          # flag as oncogenic if score >= cutoff
    tp = ((pred == 1) & (y == 1)).sum()
    fn = ((pred == 0) & (y == 1)).sum()
    fp = ((pred == 1) & (y == 0)).sum()
    tn = ((pred == 0) & (y == 0)).sum()

    rows.append({
        "tool": c,
        "recall":      round(tp / (tp + fn), 3),          # of oncogenic, fraction caught
        "specificity": round(tn / (tn + fp), 3),          # of benign, fraction correctly cleared
        "precision":   round(tp / (tp + fp), 3) if (tp + fp) else float("nan"),
        "youden_J":    round(float((tpr - fpr)[k]), 3),
    })

rec = pd.DataFrame(rows).sort_values("recall", ascending=False).reset_index(drop=True)
print(rec)

                  tool  recall  specificity  precision  youden_J
0       VARITY_R_score   0.744        0.836      0.586     0.580
1  AlphaMissense_score   0.719        0.806      0.530     0.525
2      VARITY_ER_score   0.711        0.816      0.547     0.527
3        MutPred_score   0.702        0.745      0.595     0.446
4          VEST4_score   0.700        0.806      0.523     0.507
5          ESM1b_score   0.686        0.833      0.562     0.520
6            EVE_score   0.685        0.744      0.515     0.429
7      PrimateAI_score   0.675        0.736      0.443     0.411
8          REVEL_score   0.653        0.844      0.566     0.497


In [18]:
from sklearn.metrics import roc_curve

# Work on complete cases so every variant has all 9 tools (fair "how many caught it" count)
cc = lab.dropna(subset=tool_cols).copy()
y = cc["label"].values

# 1. Give each tool a Youden cutoff, then a binary "flagged oncogenic"
calls = pd.DataFrame(index=cc.index)
for c in tool_cols:
    s = cc[c].values.astype(float)
    if c == "ESM1b_score":
        s = -s
    fpr, tpr, thr = roc_curve(y, s)
    t = thr[(tpr - fpr).argmax()]
    calls[c] = (s >= t).astype(int)

# 2. How many of the 9 tools flagged each variant
calls["n_flag"] = calls[tool_cols].sum(axis=1)
calls["label"] = y

# 3. Among true oncogenic variants: how many tools caught each?
onco = calls[calls["label"] == 1]
print("Oncogenic variants by number of tools that caught them:")
print(onco["n_flag"].value_counts().sort_index())

# 4. The headline numbers
print("\ncomplete-case oncogenic:", len(onco))
print("missed by ALL tools (n_flag == 0):", (onco["n_flag"] == 0).sum())
recalls = {c: calls.loc[calls.label == 1, c].mean() for c in tool_cols}
best = max(recalls, key=recalls.get)
print(f"best single tool recall: {best} = {recalls[best]:.3f}")
print(f"union recall (any tool flags): {(onco['n_flag'] >= 1).mean():.3f}")

Oncogenic variants by number of tools that caught them:
n_flag
0     48
1     24
2     24
3     26
4     20
5     33
6     35
7     45
8     79
9    207
Name: count, dtype: int64

complete-case oncogenic: 541
missed by ALL tools (n_flag == 0): 48
best single tool recall: VARITY_ER_score = 0.747
union recall (any tool flags): 0.911


- **Scored:** the tool produced a number for the variant. The complete-case filter guarantees all 9 tools scored all 541 oncogenic variants, so every tool had an opportunity on every variant.
- **Caught:** the tool's score cleared its Youden cutoff, meaning the tool actually calls the variant oncogenic. Having a score is not the same as calling it oncogenic. A tool can score a variant and still place it below its cutoff, which counts as calling it benign.

`n_flag` counts, per variant, how many of the 9 tools scored it **high enough to call it oncogenic**, not how many tools scored it at all. That is why it ranges from 0 to 9 rather than always being 9.

**How to read the distribution:**
- `n_flag = 9` (207 variants): every tool scored it above its cutoff. Easy consensus, all correct.
- `n_flag = 0` (48 variants): every tool scored it, but every score fell below the cutoff. All 9 tools call these truly oncogenic variants benign. 

- middle values (about 286 variants): caught by some tools but not others. This is the complementary region, where different tools catch different variants.

**Key comparison:** the best single tool catches 74.7 percent (VARITY_ER), but flagging by any tool reaches 91.1 percent. That gap exists only because the variants one tool misses are often caught by another, so their blind spots are complementary.

## 8. Weighted combination (logistic regression)

Combines all tool scores into one prediction by learning a weight per tool, evaluated with 5-fold cross-validation so the estimate is honest rather than overfit. Compared against the best single tools on the same set.

Significance: the combination improves on the best single tool. Cross-validated AUROC 0.867 and recall 0.767, versus 0.860 / 0.738 for VARITY_R and 0.821 / 0.665 for AlphaMissense. 

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve

# logistic regression needs no missing values -> use complete cases (all 9 tools present)
cc = lab.dropna(subset=tool_cols).copy()
X = cc[tool_cols].values
y = cc["label"].values
print("model data:", X.shape, "| pos:", int(y.sum()), "neg:", int((y==0).sum()))

# standardize features, then logistic regression; class_weight balances the classes (helps recall)
model = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=1000, class_weight="balanced"))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# honest out-of-fold predicted probabilities
proba = cross_val_predict(model, X, y, cv=cv, method="predict_proba")[:, 1]

# metrics for the combined model
auroc = roc_auc_score(y, proba)
fpr, tpr, thr = roc_curve(y, proba)
k = (tpr - fpr).argmax(); t = thr[k]
pred = (proba >= t).astype(int)
recall      = ((pred==1)&(y==1)).sum() / (y==1).sum()
specificity = ((pred==0)&(y==0)).sum() / (y==0).sum()

print(f"\nLogistic regression (5-fold CV): AUROC {auroc:.3f} | recall {recall:.3f} | specificity {specificity:.3f}")

# best single tools on the SAME set, for comparison
print("\nBest single tools (same complete-case set):")
for c in ["VARITY_R_score", "AlphaMissense_score"]:
    s = cc[c].values.astype(float)
    fp_, tp_, th_ = roc_curve(y, s); kk = (tp_-fp_).argmax(); tt = th_[kk]
    r = ((s>=tt)&(y==1)).sum() / (y==1).sum()
    print(f"  {c}: AUROC {roc_auc_score(y, s):.3f} | recall {r:.3f}")

model data: (1324, 9) | pos: 541 neg: 783

Logistic regression (5-fold CV): AUROC 0.867 | recall 0.767 | specificity 0.835

Best single tools (same complete-case set):
  VARITY_R_score: AUROC 0.860 | recall 0.738
  AlphaMissense_score: AUROC 0.821 | recall 0.665


## 9. Learned tool weights

Shows the weight logistic regression assigned to each tool (standardized, so magnitudes are comparable). Larger magnitude means more influence on the combined prediction.

Significance: ESM1b's negative weight is expected, since its scale is inverted. VARITY_ER's large negative weight is a collinearity artifact. VARITY_R and VARITY_ER correlate at 0.95, so the model splits their shared signal into offsetting positive and negative weights. This motivates dropping one VARITY before interpreting the weights.

In [20]:
Xs = StandardScaler().fit_transform(X)
lr = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xs, y)
weights = pd.Series(lr.coef_[0], index=tool_cols).sort_values(key=abs, ascending=False)
print("Tool weights (standardized; larger magnitude = more influence):")
print(weights.round(3))

Tool weights (standardized; larger magnitude = more influence):
VARITY_R_score         1.641
VARITY_ER_score       -0.872
MutPred_score          0.452
ESM1b_score           -0.358
PrimateAI_score       -0.218
AlphaMissense_score    0.154
VEST4_score            0.152
REVEL_score            0.116
EVE_score              0.056
dtype: float64
